# GenAI POC: Menu Item Pattern Detection

This Snowflake notebook uses Snowflake Cortex `AI_CLASSIFY` to identify whether a portfolio listing looks like:
- a standalone drink
- a meal or combo that includes a drink
- unclear / needs review

The motivation is that `item_name` is fuzzy across platforms: some rows are true drink listings, while others are meals or bundles whose drink component is only visible when `addon_text`, category fields, description, and price are considered together.

## Controls

- Cost management: classify only a sample or one market at a time.
- Determinism: use fixed categories and `AI_CLASSIFY` instead of open-ended completion.
- Human-in-the-loop: persist results into an audit table with empty review columns.
- Evaluation: compare Cortex labels against manual reviewer labels after annotation.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

SAMPLE_SIZE = 500
MARKET_FILTER = None  # e.g. 'GBR', 'DEU', 'USA' or None for all available markets
TARGET_TABLE = 'REDBULL_DATA.AUDIT.MENU_ITEM_PATTERN_POC'

In [ ]:
market_predicate = "1=1" if MARKET_FILTER is None else f"market = '{MARKET_FILTER}'"

source_sql = f"""
WITH source_rows AS (
    SELECT
        market,
        source_file,
        file_row_number,
        id_outlet,
        id_beverage,
        id_drink,
        item_name,
        addon_text,
        menu_category,
        drink_category_primary,
        drink_category_secondary,
        item_description,
        item_price,
        item_brand,
        item_subbrand,
        has_addon_prompt,
        CONCAT(
            'ITEM_NAME: ', COALESCE(item_name, ''),
            ' | ADDON_TEXT: ', COALESCE(addon_text, ''),
            ' | MENU_CATEGORY: ', COALESCE(menu_category, ''),
            ' | DRINK_CATEGORY_PRIMARY: ', COALESCE(drink_category_primary, ''),
            ' | DRINK_CATEGORY_SECONDARY: ', COALESCE(drink_category_secondary, ''),
            ' | ITEM_DESCRIPTION: ', COALESCE(item_description, ''),
            ' | ITEM_PRICE: ', COALESCE(TO_VARCHAR(item_price), ''),
            ' | BRAND: ', COALESCE(item_brand, ''),
            ' | SUBBRAND: ', COALESCE(item_subbrand, ''),
            ' | HAS_ADDON_PROMPT: ', COALESCE(TO_VARCHAR(has_addon_prompt), '')
        ) AS classification_text
    FROM REDBULL_DATA.SILVER.int_portfolio_valid
    WHERE {market_predicate}
      AND COALESCE(item_name, addon_text) IS NOT NULL
)
SELECT *
FROM source_rows
LIMIT {SAMPLE_SIZE}
"""

source_df = session.sql(source_sql)
source_df.limit(20).to_pandas()

## Cortex classification

The prompt stays narrow and the labels are mutually exclusive:
- `standalone_drink`
- `meal_or_combo_with_drink`
- `unclear_needs_review`

In [ ]:
classification_sql = f"""
CREATE OR REPLACE TABLE {TARGET_TABLE} AS
WITH source_rows AS (
    SELECT
        market,
        source_file,
        file_row_number,
        id_outlet,
        id_beverage,
        id_drink,
        item_name,
        addon_text,
        menu_category,
        drink_category_primary,
        drink_category_secondary,
        item_description,
        item_price,
        item_brand,
        item_subbrand,
        has_addon_prompt,
        CONCAT(
            'ITEM_NAME: ', COALESCE(item_name, ''),
            ' | ADDON_TEXT: ', COALESCE(addon_text, ''),
            ' | MENU_CATEGORY: ', COALESCE(menu_category, ''),
            ' | DRINK_CATEGORY_PRIMARY: ', COALESCE(drink_category_primary, ''),
            ' | DRINK_CATEGORY_SECONDARY: ', COALESCE(drink_category_secondary, ''),
            ' | ITEM_DESCRIPTION: ', COALESCE(item_description, ''),
            ' | ITEM_PRICE: ', COALESCE(TO_VARCHAR(item_price), ''),
            ' | BRAND: ', COALESCE(item_brand, ''),
            ' | SUBBRAND: ', COALESCE(item_subbrand, ''),
            ' | HAS_ADDON_PROMPT: ', COALESCE(TO_VARCHAR(has_addon_prompt), '')
        ) AS classification_text
    FROM REDBULL_DATA.SILVER.int_portfolio_valid
    WHERE {market_predicate}
      AND COALESCE(item_name, addon_text) IS NOT NULL
    LIMIT {SAMPLE_SIZE}
)
SELECT
    market,
    source_file,
    file_row_number,
    id_outlet,
    id_beverage,
    id_drink,
    item_name,
    addon_text,
    menu_category,
    drink_category_primary,
    drink_category_secondary,
    item_description,
    item_price,
    item_brand,
    item_subbrand,
    has_addon_prompt,
    classification_text,
    AI_CLASSIFY(
        classification_text,
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT(
                'label', 'standalone_drink',
                'description', 'The listing itself is a beverage item rather than a meal, bundle, or combo.'
            ),
            OBJECT_CONSTRUCT(
                'label', 'meal_or_combo_with_drink',
                'description', 'The listing is a food item, meal, menu, combo, box, bucket, bundle, or snack offer where a drink is only one component of the offer or is selected through addon text.'
            ),
            OBJECT_CONSTRUCT(
                'label', 'unclear_needs_review',
                'description', 'The listing text does not clearly indicate whether it is a standalone drink or a meal/combo.'
            )
        ),
        OBJECT_CONSTRUCT(
            'task_description', 'Classify each menu listing using item name, menu category, drink categories, description, price, and addon text. Use standalone_drink only when the sold item itself is clearly a beverage listing. Use meal_or_combo_with_drink when the sold item is food, a meal, a menu, a combo, a box, a bucket, a snack item, or another non-drink item that includes a drink or drink choice in addon text or description, even if the item name does not contain words like menu or combo. Use unclear_needs_review only when the evidence is genuinely mixed.',
            'examples', ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT(
                    'input', 'ITEM_NAME: Red Bull 250ml | ADDON_TEXT:  | MENU_CATEGORY: Drinks | DRINK_CATEGORY_PRIMARY: Energy Drinks | DRINK_CATEGORY_SECONDARY: Red Bull | ITEM_DESCRIPTION: 250ml can | ITEM_PRICE: 2.99 | BRAND: Red Bull | SUBBRAND: Red Bull Energy Drink | HAS_ADDON_PROMPT: false',
                    'labels', ARRAY_CONSTRUCT('standalone_drink'),
                    'explanation', 'This row is directly a beverage listing.'
                ),
                OBJECT_CONSTRUCT(
                    'input', 'ITEM_NAME: Cheeseburger Menu | ADDON_TEXT: choose your drink | MENU_CATEGORY: Burgers | DRINK_CATEGORY_PRIMARY: Soft Drinks | DRINK_CATEGORY_SECONDARY: Cola | ITEM_DESCRIPTION: burger, fries and a drink | ITEM_PRICE: 10.90 | BRAND: Coca-Cola | SUBBRAND: Coca-Cola Unspecified | HAS_ADDON_PROMPT: true',
                    'labels', ARRAY_CONSTRUCT('meal_or_combo_with_drink'),
                    'explanation', 'The listing is a meal/menu and the drink appears only as part of the bundle choice.'
                ),
                OBJECT_CONSTRUCT(
                    'input', 'ITEM_NAME: L.A. Chicken-Crossies (8 Stuck) | ADDON_TEXT: Coca-Cola Zero Sugar 1.0l reusable | MENU_CATEGORY: Fingerfood | DRINK_CATEGORY_PRIMARY: Soft Drink | DRINK_CATEGORY_SECONDARY: Cola | ITEM_DESCRIPTION: crispy chicken snack with selectable drink | ITEM_PRICE: 8.90 | BRAND: Coca-Cola | SUBBRAND: Coca-Cola Zero Sugar | HAS_ADDON_PROMPT: true',
                    'labels', ARRAY_CONSTRUCT('meal_or_combo_with_drink'),
                    'explanation', 'The sold item is fingerfood, not a beverage. The drink appears only as an included or selected component, so it should be treated as a meal/combo with drink.'
                ),
                OBJECT_CONSTRUCT(
                    'input', 'ITEM_NAME: Family Box | ADDON_TEXT: optional drink selection | MENU_CATEGORY: Specials | DRINK_CATEGORY_PRIMARY:  | DRINK_CATEGORY_SECONDARY:  | ITEM_DESCRIPTION: large sharing menu with optional drink add-on | ITEM_PRICE: 24.90 | BRAND: Sprite | SUBBRAND: Sprite Unspecified | HAS_ADDON_PROMPT: true',
                    'labels', ARRAY_CONSTRUCT('unclear_needs_review'),
                    'explanation', 'The row may be a bundle with drink options, but the text is not specific enough to classify confidently.'
                )
            )
        )
    ):labels[0]::STRING AS cortex_label,
    NULL::STRING AS manual_review_label,
    NULL::STRING AS manual_review_notes,
    CURRENT_TIMESTAMP() AS scored_at
FROM source_rows
"""

session.sql(classification_sql).collect()

In [ ]:
session.sql(f"""
SELECT cortex_label, COUNT(*) AS row_count
FROM {TARGET_TABLE}
GROUP BY cortex_label
ORDER BY row_count DESC
""").to_pandas()

In [ ]:
session.sql(f"""
SELECT market, item_name, addon_text, menu_category, drink_category_primary, drink_category_secondary, item_price, cortex_label
FROM {TARGET_TABLE}
ORDER BY scored_at DESC
LIMIT 50
""").to_pandas()

## Production cost management

If deployed beyond a proof of concept, cost should be controlled through scope reduction and incremental processing rather than sending every row to Cortex.

Recommended production pattern:
- Apply cheap SQL heuristics first and only route ambiguous rows to Cortex.
- Score only new or changed rows instead of rescoring the full history.
- Keep the prompt compact and only include fields that materially improve accuracy.
- Persist predictions so the same row is not scored repeatedly.
- Add row-count or budget limits per run to prevent unexpected spend spikes.

Practical examples:
- obvious drinks can be classified by rule without AI
- obvious meal/menu rows can also be classified by rule
- Cortex should focus on the ambiguous middle cases where text interpretation adds value

This means the production system should be hybrid:
- rule-based classification for easy cases
- Cortex classification for uncertain cases
- human review for low-confidence or business-critical outputs


## Human review and evaluation

Recommended workflow:
1. Review a sample of `cortex_label` outputs.
2. Populate `manual_review_label` and `manual_review_notes` for a reviewed subset.
3. Measure agreement between Cortex and reviewer labels.

Suggested evaluation metrics:
- overall agreement rate
- precision / recall for `meal_or_combo_with_drink`
- review rate for `unclear_needs_review`
- cost per 1000 classified rows

Production notes:
- Keep labels mutually exclusive to improve accuracy.
- Run on new or changed rows only to control cost.
- Grant the execution role `SNOWFLAKE.CORTEX_USER` before running.
- `AI_CLASSIFY` is preferred here over free-form completion because it gives constrained labels and more stable output.

Reference:
- Snowflake docs: `AI_CLASSIFY` is the latest text classification function and requires the `SNOWFLAKE.CORTEX_USER` database role.
- Snowflake docs also note that descriptions/examples improve classification quality but increase token cost.